# 00_runtime_mode_and_user_config

In [ ]:
from __future__ import annotations
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

RUN_PROFILE = os.environ.get('TSTW_RUN_PROFILE', 'real_video_vae_latent_probe_completion_formal').strip()
RUN_MODE = 'formal'
REQUIRE_FORMAL_PASS = True
WORKFLOW_KEY = 'detection_evaluation'
STEP_KEY = 'step02_run_real_video_vae_latent_probe'
FAMILY_ID_INPUT = os.environ.get('TSTW_FAMILY_ID', '').strip()

DRIVE_ROOT = Path('/content/drive/MyDrive')
TSTW_ROOT = DRIVE_ROOT / 'TSTW'
FAMILY_RESULTS_ROOT = TSTW_ROOT / 'results' / 'families'
OVERRIDE_ROOT = TSTW_ROOT / 'configs' / RUN_PROFILE

LOCAL_TSTW_ROOT = Path('/content/TSTW_runtime')
LOCAL_REPO_DIR = LOCAL_TSTW_ROOT / 'repo'
LOCAL_DATASET_CACHE_DIR = LOCAL_TSTW_ROOT / 'dataset_cache'
LOCAL_DATASET_DIR = LOCAL_TSTW_ROOT / 'datasets' / 'real_video_probe'
LOCAL_SESSION_MODELS_DIR = LOCAL_TSTW_ROOT / 'session_models'
LOCAL_MODEL_CACHE_DIR = LOCAL_TSTW_ROOT / 'model_cache'
LOCAL_RUNS_DIR = LOCAL_TSTW_ROOT / 'runs'
LOCAL_TMP_DIR = LOCAL_TSTW_ROOT / 'tmp'

DATASET_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'dataset_registry.json'
MODEL_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'model_registry.json'
DRIVE_STATE_PATH = TSTW_ROOT / 'registry' / 'drive_state.json'
RUNTIME_OVERRIDE_PATH = OVERRIDE_ROOT / 'runtime_override.json'
DATASET_OVERRIDE_PATH = OVERRIDE_ROOT / 'dataset_override.json'
MODEL_OVERRIDE_PATH = OVERRIDE_ROOT / 'model_override.json'
RESULT_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'result_registry.jsonl'
FAMILY_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'family_registry.jsonl'

REPO_URL = os.environ.get('TSTW_VW_REPO_URL', 'https://github.com/your-org/TSTW-VW.git').strip()
REPO_BRANCH = os.environ.get('TSTW_VW_REPO_BRANCH', 'main').strip()

if not REPO_URL:
    raise ValueError('TSTW_VW_REPO_URL is required')

print({
    'run_profile': RUN_PROFILE,
    'run_mode': RUN_MODE,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'family_id_input': FAMILY_ID_INPUT or None,
    'repo_url': REPO_URL,
    'repo_branch': REPO_BRANCH,
    'drive_root': str(DRIVE_ROOT),
    'local_repo_dir': str(LOCAL_REPO_DIR),
})

# 01_mount_google_drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 02_read_drive_state_and_overrides

In [ ]:
for registry_path in (DATASET_REGISTRY_PATH, MODEL_REGISTRY_PATH, DRIVE_STATE_PATH):
    if not registry_path.exists():
        raise FileNotFoundError(registry_path)
for override_path in (RUNTIME_OVERRIDE_PATH, DATASET_OVERRIDE_PATH, MODEL_OVERRIDE_PATH):
    if not override_path.exists():
        raise FileNotFoundError(override_path)

dataset_registry = json.loads(DATASET_REGISTRY_PATH.read_text(encoding='utf-8'))
model_registry = json.loads(MODEL_REGISTRY_PATH.read_text(encoding='utf-8'))
drive_state = json.loads(DRIVE_STATE_PATH.read_text(encoding='utf-8'))
runtime_override = json.loads(RUNTIME_OVERRIDE_PATH.read_text(encoding='utf-8'))
dataset_override = json.loads(DATASET_OVERRIDE_PATH.read_text(encoding='utf-8'))
model_override = json.loads(MODEL_OVERRIDE_PATH.read_text(encoding='utf-8'))

dataset_key = str(dataset_override.get('dataset_key') or drive_state.get('default_dataset_key') or '').strip()
if not dataset_key:
    raise ValueError('dataset_key is required in dataset_override or drive_state')

def _sanitize_fragment(value: str, fallback: str) -> str:
    cleaned = ''.join(character for character in value if character.isalnum() or character in ['_', '-']).strip('_-')
    return cleaned[:64] if cleaned else fallback

utc_time = dt.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
dataset_fragment = _sanitize_fragment(dataset_key, 'dataset')
family_scope = 'stage02_real_video_vae_latent_probe'
if FAMILY_ID_INPUT:
    FAMILY_ID = FAMILY_ID_INPUT
else:
    FAMILY_ID = f"{family_scope}__{RUN_MODE}__{dataset_fragment}__{utc_time}__nogit"

FAMILY_ROOT = FAMILY_RESULTS_ROOT / FAMILY_ID
STEP_ROOT = FAMILY_ROOT / 'steps' / STEP_KEY
STEP_PACKAGES_ROOT = STEP_ROOT / 'packages'
STEP_CHECKS_ROOT = STEP_ROOT / 'checks'
STEP_SUMMARIES_ROOT = STEP_ROOT / 'summaries'
STEP_LOGS_ROOT = STEP_ROOT / 'logs'

runtime_override['run_profile'] = RUN_PROFILE
runtime_override['run_mode'] = RUN_MODE
runtime_override['workflow_key'] = WORKFLOW_KEY
runtime_override['step_key'] = STEP_KEY
runtime_override['family_id'] = FAMILY_ID
runtime_override['repo_url'] = REPO_URL
runtime_override['repo_branch'] = REPO_BRANCH
runtime_override['dataset_registry_path'] = str(DATASET_REGISTRY_PATH)
runtime_override['model_registry_path'] = str(MODEL_REGISTRY_PATH)
runtime_override['result_registry_path'] = str(RESULT_REGISTRY_PATH)
runtime_override['family_registry_path'] = str(FAMILY_REGISTRY_PATH)

print({
    'dataset_registry': str(DATASET_REGISTRY_PATH),
    'model_registry': str(MODEL_REGISTRY_PATH),
    'drive_state': str(DRIVE_STATE_PATH),
    'runtime_override': str(RUNTIME_OVERRIDE_PATH),
    'dataset_override': str(DATASET_OVERRIDE_PATH),
    'model_override': str(MODEL_OVERRIDE_PATH),
    'dataset_key': dataset_key,
    'family_id': FAMILY_ID,
    'family_root': str(FAMILY_ROOT),
    'step_root': str(STEP_ROOT),
})

# 03_prepare_local_workspace

In [ ]:
LOCAL_TSTW_ROOT.mkdir(parents=True, exist_ok=True)
for runtime_path in (
    LOCAL_DATASET_CACHE_DIR,
    LOCAL_DATASET_DIR,
    LOCAL_SESSION_MODELS_DIR,
    LOCAL_MODEL_CACHE_DIR,
    LOCAL_RUNS_DIR,
    LOCAL_TMP_DIR,
):
    if runtime_path.exists():
        shutil.rmtree(runtime_path)
    runtime_path.mkdir(parents=True, exist_ok=True)

if LOCAL_REPO_DIR.exists():
    shutil.rmtree(LOCAL_REPO_DIR)
LOCAL_REPO_DIR.mkdir(parents=True, exist_ok=True)

for drive_dir in (
    FAMILY_RESULTS_ROOT,
    FAMILY_ROOT,
    STEP_ROOT,
    STEP_PACKAGES_ROOT,
    STEP_CHECKS_ROOT,
    STEP_SUMMARIES_ROOT,
    STEP_LOGS_ROOT,
    RESULT_REGISTRY_PATH.parent,
    FAMILY_REGISTRY_PATH.parent,
    TSTW_ROOT / 'configs',
):
    drive_dir.mkdir(parents=True, exist_ok=True)

print({
    'local_root': str(LOCAL_TSTW_ROOT),
    'local_repo_dir': str(LOCAL_REPO_DIR),
    'local_session_models_dir': str(LOCAL_SESSION_MODELS_DIR),
    'local_model_cache_dir': str(LOCAL_MODEL_CACHE_DIR),
    'family_root': str(FAMILY_ROOT),
    'step_root': str(STEP_ROOT),
})

# 04_clone_or_update_repository

In [ ]:
if (LOCAL_REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if any(LOCAL_REPO_DIR.iterdir()):
        shutil.rmtree(LOCAL_REPO_DIR)
        LOCAL_REPO_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(LOCAL_REPO_DIR)], check=True)
git_commit = subprocess.check_output(['git', '-C', str(LOCAL_REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
git_status_short = subprocess.check_output(['git', '-C', str(LOCAL_REPO_DIR), 'status', '--short'], text=True).strip()
short_commit = git_commit[:7]
print({'git_commit': git_commit, 'short_commit': short_commit, 'has_uncommitted_changes': bool(git_status_short)})

# 05_install_dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(LOCAL_REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', 'diffusers', 'accelerate', 'transformers', 'safetensors', 'opencv-python', 'imageio', 'imageio-ffmpeg', 'ffmpeg-python', 'numpy', 'pandas', 'scipy', 'scikit-image', 'tqdm', 'lpips', 'pytorch-msssim'], check=True)
pip_freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
print({'pip_freeze_lines': len([line for line in pip_freeze.splitlines() if line.strip()])})

# 06_copy_and_validate_dataset

In [ ]:
def _sha256_file(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with file_path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest().lower()


def _collect_dataset_tree_hints(root: Path, max_items: int = 30) -> dict:
    if not root.exists():
        return {'exists': False, 'sample_files': [], 'sample_dirs': []}
    sample_files = []
    for file_path in sorted(path for path in root.rglob('*') if path.is_file())[:max_items]:
        sample_files.append(str(file_path.relative_to(root)).replace('\\', '/'))
    sample_dirs = []
    for dir_path in sorted(path for path in root.rglob('*') if path.is_dir())[:max_items]:
        sample_dirs.append(str(dir_path.relative_to(root)).replace('\\', '/'))
    return {'exists': True, 'sample_files': sample_files, 'sample_dirs': sample_dirs}


def _resolve_cache_tar_path(dataset_override_payload: dict) -> Path:
    cache_tar_raw = str(dataset_override_payload.get('cache_tar_path', '')).strip()
    if not cache_tar_raw:
        raise RuntimeError(
            {
                'reason': 'dataset_cache_tar_path_missing',
                'expected_field': 'dataset_override.cache_tar_path',
                'recommended_example': '/content/drive/MyDrive/TSTW/datasets/cache/<dataset_key>.tar.zst',
            }
        )
    candidate = Path(cache_tar_raw)
    if candidate.is_absolute():
        return candidate

    # 支持相对路径写法：优先相对 TSTW_ROOT，其次相对 datasets/cache。
    relative_candidates = [
        TSTW_ROOT / cache_tar_raw,
        TSTW_ROOT / 'datasets' / 'cache' / cache_tar_raw,
    ]
    for resolved in relative_candidates:
        if resolved.exists():
            return resolved

    # 返回首选候选，后续统一抛可解释错误。
    return relative_candidates[0]


def _raise_dataset_error(reason: str, payload: dict) -> None:
    raise RuntimeError({'reason': reason, **payload})


def _validate_manifest_payload(manifest_payload: dict, dataset_root: Path) -> dict:
    if not isinstance(manifest_payload, dict):
        _raise_dataset_error('dataset_manifest_not_dict', {'dataset_root': str(dataset_root)})

    samples = manifest_payload.get('samples')
    if not isinstance(samples, list) or not samples:
        _raise_dataset_error(
            'dataset_manifest_samples_invalid',
            {
                'dataset_root': str(dataset_root),
                'expected': 'manifest.samples must be a non-empty list',
            },
        )

    missing_relpaths = []
    absolute_relpaths = []
    for sample in samples:
        relpath = str(sample.get('relpath', '')).strip()
        if not relpath:
            continue
        relpath_obj = Path(relpath)
        if relpath_obj.is_absolute():
            absolute_relpaths.append(relpath)
            continue
        resolved = dataset_root / relpath_obj
        if not resolved.exists():
            missing_relpaths.append(relpath)

    if absolute_relpaths:
        _raise_dataset_error(
            'dataset_manifest_relpath_must_be_relative',
            {
                'dataset_root': str(dataset_root),
                'absolute_relpaths': absolute_relpaths[:20],
            },
        )

    if missing_relpaths:
        _raise_dataset_error(
            'dataset_manifest_relpath_missing_files',
            {
                'dataset_root': str(dataset_root),
                'missing_relpaths': missing_relpaths[:50],
                'tree_hint': _collect_dataset_tree_hints(dataset_root),
            },
        )

    return {'sample_count': len(samples), 'missing_relpath_count': len(missing_relpaths)}


cache_tar_path = _resolve_cache_tar_path(dataset_override)
cache_tar_sha256 = str(dataset_override.get('cache_tar_sha256', '')).strip().lower()
if not cache_tar_path.exists():
    _raise_dataset_error(
        'dataset_cache_tar_not_found',
        {
            'cache_tar_path': str(cache_tar_path),
            'dataset_key': dataset_key,
            'recommended_location': str(TSTW_ROOT / 'datasets' / 'cache'),
        },
    )

if cache_tar_path.suffixes[-2:] != ['.tar', '.zst'] and cache_tar_path.suffix != '.zst':
    _raise_dataset_error(
        'dataset_cache_tar_suffix_invalid',
        {
            'cache_tar_path': str(cache_tar_path),
            'expected_suffix': '.tar.zst',
        },
    )

local_tar_path = LOCAL_DATASET_CACHE_DIR / cache_tar_path.name
shutil.copy2(cache_tar_path, local_tar_path)

actual_sha256 = _sha256_file(local_tar_path)
if cache_tar_sha256 and actual_sha256 != cache_tar_sha256:
    _raise_dataset_error(
        'dataset_cache_tar_sha256_mismatch',
        {
            'expected_sha256': cache_tar_sha256,
            'actual_sha256': actual_sha256,
            'local_tar_path': str(local_tar_path),
        },
    )

subprocess.run(['tar', '--zstd', '-xf', str(local_tar_path), '-C', str(LOCAL_DATASET_DIR)], check=True)

local_manifest_candidates = sorted(LOCAL_DATASET_DIR.rglob('dataset_manifest.json'))
if not local_manifest_candidates:
    _raise_dataset_error(
        'dataset_manifest_not_found_after_extract',
        {
            'local_dataset_root': str(LOCAL_DATASET_DIR),
            'tree_hint': _collect_dataset_tree_hints(LOCAL_DATASET_DIR),
        },
    )

local_dataset_manifest_path = local_manifest_candidates[0]
manifest_payload = json.loads(local_dataset_manifest_path.read_text(encoding='utf-8'))
manifest_stats = _validate_manifest_payload(manifest_payload, LOCAL_DATASET_DIR)

local_mp4_paths = list(LOCAL_DATASET_DIR.rglob('*.mp4'))
local_mp4_count = len(local_mp4_paths)
if local_mp4_count < 1:
    _raise_dataset_error(
        'dataset_mp4_not_found_after_extract',
        {
            'local_dataset_root': str(LOCAL_DATASET_DIR),
            'dataset_manifest_path': str(local_dataset_manifest_path),
            'tree_hint': _collect_dataset_tree_hints(LOCAL_DATASET_DIR),
        },
    )

runtime_override['local_dataset_root'] = str(LOCAL_DATASET_DIR)
runtime_override['dataset_manifest_path'] = str(local_dataset_manifest_path)
runtime_override['dataset_cache_tar_local_path'] = str(local_tar_path)
runtime_override['dataset_cache_tar_sha256_actual'] = actual_sha256
runtime_override['dataset_key'] = dataset_key

print({
    'dataset_tar_drive_path': str(cache_tar_path),
    'dataset_tar_local_path': str(local_tar_path),
    'dataset_tar_sha256_actual': actual_sha256,
    'dataset_manifest_path': str(local_dataset_manifest_path),
    'manifest_sample_count': manifest_stats['sample_count'],
    'mp4_count': local_mp4_count,
    'local_dataset_root': str(LOCAL_DATASET_DIR),
})

# 07_copy_and_validate_models

In [ ]:
os.environ['HF_HOME'] = str(LOCAL_MODEL_CACHE_DIR / 'huggingface')
os.environ['TRANSFORMERS_CACHE'] = str(LOCAL_MODEL_CACHE_DIR / 'huggingface' / 'transformers')
os.environ['TORCH_HOME'] = str(LOCAL_MODEL_CACHE_DIR / 'torch')

(LOCAL_MODEL_CACHE_DIR / 'huggingface').mkdir(parents=True, exist_ok=True)
(LOCAL_MODEL_CACHE_DIR / 'torch').mkdir(parents=True, exist_ok=True)

from huggingface_hub import snapshot_download
import lpips

def _resolve_model_repo(config_payload: dict, role: str, fallback_repo: str, fallback_revision: str = 'main') -> tuple[str, str]:
    role_payload = config_payload.get(role, {}) if isinstance(config_payload, dict) else {}
    if isinstance(role_payload, dict):
        repo_id = str(role_payload.get('repo_id') or role_payload.get('model_id') or fallback_repo).strip()
        revision = str(role_payload.get('revision') or fallback_revision).strip()
    else:
        repo_id = fallback_repo
        revision = fallback_revision
    if not repo_id:
        raise ValueError(f'{role} repo_id is required')
    return repo_id, revision

def _download_hf_snapshot(model_role: str, repo_id: str, revision: str) -> Path:
    local_model_dir = LOCAL_SESSION_MODELS_DIR / model_role / repo_id.replace('/', '__')
    local_model_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=repo_id,
        revision=revision,
        local_dir=str(local_model_dir),
        local_dir_use_symlinks=False,
    )
    return local_model_dir

vae_repo_id, vae_revision = _resolve_model_repo(
    model_override,
    role='vae',
    fallback_repo='stabilityai/sd-vae-ft-mse',
)

vae_local_model_root = _download_hf_snapshot('stage2_vae', vae_repo_id, vae_revision)
lpips_model = lpips.LPIPS(net='alex')
del lpips_model

runtime_override['local_vae_model_root'] = str(vae_local_model_root)
runtime_override['local_lpips_model_root'] = str(LOCAL_MODEL_CACHE_DIR / 'torch')
runtime_override['local_lpips_weight_path'] = None
runtime_override['local_clip_model_root'] = None
runtime_override['allow_mock_vae_backend'] = False
runtime_override['model_policy'] = 'session_only_no_drive_model_storage'
runtime_override['model_layout_digest'] = {
    'vae_local_model_root': str(vae_local_model_root),
    'cache_root': str(LOCAL_MODEL_CACHE_DIR),
}

session_model_manifest_payload = {
    'model_policy': 'session_only_no_drive_model_storage',
    'models': [
        {
            'model_role': 'stage2_vae',
            'repo_id': vae_repo_id,
            'revision': vae_revision,
            'local_path': str(vae_local_model_root),
            'load_api': 'AutoencoderKL.from_pretrained',
            'saved_to_drive': False,
            'included_in_result_package': False,
        },
        {
            'model_role': 'lpips_quality_metric',
            'source': 'lpips_and_torchvision_runtime_cache',
            'allow_torchvision_backbone_auto_download': True,
            'local_cache_policy': 'session_only',
            'local_path': str(LOCAL_MODEL_CACHE_DIR / 'torch'),
            'saved_to_drive': False,
            'included_in_result_package': False,
        },
    ],
}

print({
    'vae_repo_id': vae_repo_id,
    'vae_revision': vae_revision,
    'local_vae_model_root': str(vae_local_model_root),
    'local_lpips_cache': str(LOCAL_MODEL_CACHE_DIR / 'torch'),
    'model_policy': session_model_manifest_payload['model_policy'],
})

# 08_check_gpu_and_runtime

In [ ]:
sys.path.insert(0, str(LOCAL_REPO_DIR))
from paper_workflow.colab_utils.runtime_check import run_runtime_preflight_check
runtime_check_report = run_runtime_preflight_check(run_mode=RUN_MODE, local_dataset_dir=LOCAL_DATASET_DIR, local_model_dirs=[Path(local_vae_model_dir), Path(local_lpips_model_dir)])
runtime_override['runtime_check_report'] = runtime_check_report
print(runtime_check_report)

# 09_verify_repository_contract

In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], cwd=LOCAL_REPO_DIR, check=True)
audit_output = subprocess.check_output([sys.executable, 'tools/harness/run_all_audits.py'], cwd=LOCAL_REPO_DIR, text=True)
print(audit_output[-1000:] if len(audit_output) > 1000 else audit_output)

# 10_run_unit_tests_smoke

In [ ]:
pytest_output = subprocess.check_output([sys.executable, '-m', 'pytest', '-q', '-m', 'smoke', 'tests/test_real_video_vae_latent_records_schema.py', 'tests/test_real_video_vae_latent_table_rebuild.py', 'tests/test_real_video_vae_latent_drive_packager.py', 'tests/test_real_video_vae_latent_quality_metrics.py', 'tests/test_real_video_vae_latent_temporal_metrics.py'], cwd=LOCAL_REPO_DIR, text=True)
print(pytest_output)

# 11_run_stage2_completion_formal

In [ ]:
construction_phase = 'real_video_vae_latent_probe'
utc_time = dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
dataset_short = ''.join(character for character in dataset_key if character.isalnum() or character in ['_', '-'])[:32] or 'dataset'
run_id = f"{construction_phase}__{RUN_PROFILE}__{dataset_short}__{utc_time}__{short_commit}"
run_root = LOCAL_RUNS_DIR / run_id
runtime_config_path = run_root / 'artifacts' / 'runtime_config.json'
runtime_manifest_path = run_root / 'artifacts' / 'runtime_manifest.json'
session_model_manifest_path = run_root / 'artifacts' / 'session_model_manifest.json'
run_root.mkdir(parents=True, exist_ok=True)
(run_root / 'artifacts').mkdir(parents=True, exist_ok=True)

runtime_override['run_id'] = run_id
runtime_override['git_commit'] = git_commit
runtime_override['dataset_manifest_snapshot_path'] = str(run_root / 'artifacts' / 'dataset_manifest_snapshot.json')
runtime_override['runtime_override_source'] = str(RUNTIME_OVERRIDE_PATH)
runtime_override['session_model_manifest_path'] = str(session_model_manifest_path)

session_model_manifest_path.write_text(
    json.dumps(session_model_manifest_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

runtime_override['runtime_manifest_overrides'] = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'family_root': str(FAMILY_ROOT),
    'step_root': str(STEP_ROOT),
    'repo_url': REPO_URL,
    'repo_branch': REPO_BRANCH,
    'git_status_short': git_status_short,
    'dataset_preflight': {
        'dataset_key': dataset_key,
        'dataset_tar_drive_path': str(cache_tar_path),
        'dataset_tar_local_path': str(local_tar_path),
        'dataset_tar_sha256_actual': actual_sha256,
        'dataset_manifest_path': str(local_dataset_manifest_path),
        'manifest_sample_count': manifest_stats['sample_count'],
        'mp4_count': local_mp4_count,
        'local_dataset_root': str(LOCAL_DATASET_DIR),
    },
    'model_preflight': {
        'model_policy': 'session_only_no_drive_model_storage',
        'session_model_manifest_path': str(session_model_manifest_path),
        'local_vae_model_root': str(vae_local_model_root),
        'local_lpips_cache': str(LOCAL_MODEL_CACHE_DIR / 'torch'),
        'allow_mock_vae_backend': runtime_override['allow_mock_vae_backend'],
    },
    'runtime_preflight': runtime_check_report,
    'runtime_manifest_path': str(runtime_manifest_path),
}

runtime_config_path.write_text(
    json.dumps(runtime_override, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

sys.path.insert(0, str(LOCAL_REPO_DIR))
from experiments.real_video_vae_latent_probe.runner import RealVideoVaeLatentRunner
runner = RealVideoVaeLatentRunner(LOCAL_REPO_DIR)
formal_result = runner.run(
    output_root=run_root,
    run_mode='formal',
    runtime_profile_override='formal',
    dataset_manifest_path=local_dataset_manifest_path,
    runtime_config_path=runtime_config_path,
)

print({
    'family_id': FAMILY_ID,
    'run_id': formal_result.run_id,
    'run_root': str(run_root),
    'event_records': len(formal_result.event_score_records),
    'threshold_records': len(formal_result.threshold_records),
    'session_model_manifest': str(session_model_manifest_path),
})

# 12_rebuild_tables_and_reports

In [ ]:
from experiments.real_video_vae_latent_probe.artifact_builder import RealVideoVaeLatentArtifactBuilder
rebuilt_paths = RealVideoVaeLatentArtifactBuilder().rebuild_artifacts(run_root)
dataset_manifest_src = LOCAL_REPO_DIR / 'configs' / 'data' / 'real_video_probe_manifest.json'
dataset_manifest_dst = run_root / 'artifacts' / 'dataset_manifest_snapshot.json'
shutil.copy2(dataset_manifest_src, dataset_manifest_dst)
(run_root / 'artifacts' / 'config_snapshot').mkdir(parents=True, exist_ok=True)
for config_path in (RUNTIME_OVERRIDE_PATH, DATASET_OVERRIDE_PATH, MODEL_OVERRIDE_PATH):
    shutil.copy2(config_path, run_root / 'artifacts' / 'config_snapshot' / config_path.name)
print({key: str(value) for key, value in rebuilt_paths.items()})

# 13_validate_formal_outputs

In [ ]:
from scripts.check_results.real_video_vae_latent_output_checker import check_real_video_vae_latent_outputs
formal_checks = check_real_video_vae_latent_outputs(run_root, run_mode='formal', require_formal_pass_criteria=REQUIRE_FORMAL_PASS)
checks_path_local = run_root / 'artifacts' / 'checks.json'
checks_path_local.write_text(json.dumps(formal_checks, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
runtime_manifest_payload = json.loads(runtime_manifest_path.read_text(encoding='utf-8'))
runtime_manifest_payload['formal_validation_summary'] = {
    'status': bool(formal_checks.get('status', False)),
    'decision': formal_checks.get('RealVideoVaeLatentDecision'),
    'blocking_reasons': formal_checks.get('BlockingReasons'),
    'next_allowed_stage': formal_checks.get('NextAllowedStage'),
    'require_formal_pass_criteria': REQUIRE_FORMAL_PASS,
    'checks_path': str(checks_path_local),
}
runtime_manifest_path.write_text(json.dumps(runtime_manifest_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
if not formal_checks['status']:
    raise RuntimeError(formal_checks)
print(formal_checks)

# 14_pack_run_to_drive

In [ ]:
from scripts.package_results.drive_packager import pack_real_video_vae_latent_run
from scripts.package_results.tar_zst_packager import pack_run_to_tar_zst

compat_pack_root = run_root / 'artifacts' / 'compat_pack'
compat_pack_root.mkdir(parents=True, exist_ok=True)
compat_pack = pack_real_video_vae_latent_run(
    run_root=run_root,
    drive_output_dir=compat_pack_root,
    include_records=True,
    include_thresholds=True,
    include_tables=True,
    include_figures=True,
    include_reports=True,
    include_failure_gallery=True,
    include_manifest=True,
)

STEP_PACKAGES_ROOT.mkdir(parents=True, exist_ok=True)
tar_pack = pack_run_to_tar_zst(
    run_root=run_root,
    drive_result_dir=STEP_PACKAGES_ROOT,
    checks_payload=formal_checks,
    exclude_large_intermediate_latents=True,
)

drive_archive_path = tar_pack['archive_path']
drive_summary_path = tar_pack['summary_path']
drive_checks_path = tar_pack['checks_path']

print({
    'family_id': FAMILY_ID,
    'step_key': STEP_KEY,
    'archive_path': str(drive_archive_path),
    'summary_path': str(drive_summary_path),
    'checks_path': str(drive_checks_path),
    'compat_zip_path': str(compat_pack['zip_path']),
})

# 15_update_result_registry

In [ ]:
registry_entry = {
    'schema_version': 'tstw_result_registry_entry.v1',
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'run_id': run_id,
    'run_profile': RUN_PROFILE,
    'run_mode': RUN_MODE,
    'construction_phase': construction_phase,
    'dataset_key': dataset_key,
    'processed_dataset_key': dataset_key,
    'git_commit': git_commit,
    'archive_path': str(drive_archive_path),
    'summary_path': str(drive_summary_path),
    'checks_path': str(drive_checks_path),
    'decision': formal_checks.get('RealVideoVaeLatentDecision'),
    'blocking_reasons': formal_checks.get('BlockingReasons', []),
    'created_at': dt.datetime.utcnow().isoformat() + 'Z',
}

with RESULT_REGISTRY_PATH.open('a', encoding='utf-8') as handle:
    handle.write(json.dumps(registry_entry, ensure_ascii=False) + '\n')

family_registry_entry = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'latest_step_key': STEP_KEY,
    'run_mode': RUN_MODE,
    'run_profile': RUN_PROFILE,
    'dataset_key': dataset_key,
    'git_commit': git_commit,
    'decision': formal_checks.get('RealVideoVaeLatentDecision'),
    'archive_path': str(drive_archive_path),
    'updated_at': dt.datetime.utcnow().isoformat() + 'Z',
}
with FAMILY_REGISTRY_PATH.open('a', encoding='utf-8') as handle:
    handle.write(json.dumps(family_registry_entry, ensure_ascii=False) + '\n')

print({
    'result_registry_path': str(RESULT_REGISTRY_PATH),
    'family_registry_path': str(FAMILY_REGISTRY_PATH),
    'decision': registry_entry['decision'],
})

# 16_print_final_summary

In [ ]:
log_dir = run_root / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
(log_dir / 'colab.log').write_text('Notebook completed successfully.\n', encoding='utf-8')
(log_dir / 'pytest.log').write_text(pytest_output, encoding='utf-8')
(log_dir / 'audit.log').write_text(audit_output, encoding='utf-8')
(log_dir / 'dependency_freeze.txt').write_text(pip_freeze, encoding='utf-8')
(log_dir / 'git_commit.txt').write_text(git_commit + '\n', encoding='utf-8')
(log_dir / 'git_status.txt').write_text(git_status_short + '\n', encoding='utf-8')

summary = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'run_id': run_id,
    'run_root': str(run_root),
    'decision': formal_checks.get('RealVideoVaeLatentDecision'),
    'blocking_reasons': formal_checks.get('BlockingReasons', []),
    'next_allowed_stage': formal_checks.get('NextAllowedStage'),
    'drive_archive_path': str(drive_archive_path),
    'drive_summary_path': str(drive_summary_path),
    'drive_checks_path': str(drive_checks_path),
    'result_registry_path': str(RESULT_REGISTRY_PATH),
    'family_registry_path': str(FAMILY_REGISTRY_PATH),
}

step_manifest = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'run_id': run_id,
    'run_mode': RUN_MODE,
    'run_profile': RUN_PROFILE,
    'dataset_key': dataset_key,
    'run_root': str(run_root),
    'package_path': str(drive_archive_path),
    'summary_path': str(drive_summary_path),
    'checks_path': str(drive_checks_path),
    'session_model_manifest_path': str(run_root / 'artifacts' / 'session_model_manifest.json'),
    'model_policy': 'session_only_no_drive_model_storage',
    'excluded_from_package': [
        'session_models/**',
        'model_cache/**',
        '*.pth',
        '*.pt',
        '*.ckpt',
        '*.safetensors',
        '*.bin',
    ],
}

family_manifest = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'run_mode': RUN_MODE,
    'run_profile': RUN_PROFILE,
    'dataset_key': dataset_key,
    'git_commit': git_commit,
    'family_root': str(FAMILY_ROOT),
    'steps': [STEP_KEY],
    'latest_step_manifest': str(STEP_ROOT / 'step_manifest.json'),
    'latest_result_registry_entry': registry_entry,
}

family_summary = {
    'family_id': FAMILY_ID,
    'decision': formal_checks.get('RealVideoVaeLatentDecision'),
    'blocking_reasons': formal_checks.get('BlockingReasons', []),
    'canonical_outputs': {
        'archive_path': str(drive_archive_path),
        'summary_path': str(drive_summary_path),
        'checks_path': str(drive_checks_path),
    },
    'analysis_only_outputs': {
        'compat_zip_path': str(compat_pack['zip_path']),
    },
}

family_checks = {
    'family_id': FAMILY_ID,
    'status': bool(formal_checks.get('status', False)),
    'formal_checks': formal_checks,
}

runtime_manifest_payload = json.loads(runtime_manifest_path.read_text(encoding='utf-8'))
runtime_manifest_payload['drive_result_summary'] = {
    'family_id': FAMILY_ID,
    'step_key': STEP_KEY,
    'archive_path': str(drive_archive_path),
    'summary_path': str(drive_summary_path),
    'checks_path': str(drive_checks_path),
    'result_registry_path': str(RESULT_REGISTRY_PATH),
}
runtime_manifest_payload['final_summary'] = summary
runtime_manifest_payload['result_registry_entry'] = registry_entry
runtime_manifest_payload['family_registry_entry'] = family_registry_entry
runtime_manifest_path.write_text(json.dumps(runtime_manifest_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

STEP_ROOT.mkdir(parents=True, exist_ok=True)
STEP_LOGS_ROOT.mkdir(parents=True, exist_ok=True)
STEP_SUMMARIES_ROOT.mkdir(parents=True, exist_ok=True)
STEP_CHECKS_ROOT.mkdir(parents=True, exist_ok=True)

(STEP_ROOT / 'step_manifest.json').write_text(json.dumps(step_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(STEP_SUMMARIES_ROOT / 'step_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(STEP_CHECKS_ROOT / 'step_checks.json').write_text(json.dumps(formal_checks, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(STEP_LOGS_ROOT / 'colab_summary.log').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

(FAMILY_ROOT / 'family_manifest.json').write_text(json.dumps(family_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(FAMILY_ROOT / 'family_summary.json').write_text(json.dumps(family_summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(FAMILY_ROOT / 'family_checks.json').write_text(json.dumps(family_checks, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

(run_root / 'reports' / 'colab_final_summary.md').write_text(
    '\n'.join([f'- {key}: {value}' for key, value in summary.items()]) + '\n',
    encoding='utf-8',
)
print(json.dumps(summary, ensure_ascii=False, indent=2))